
###Esta celda recorre todos los archivos ZIP en la carpeta bronze_vuelos/Dataset, extrae el archivo CSV interno de cada ZIP y lo guarda con un nombre único basado en el nombre del ZIP. Así, cada archivo CSV queda mapeado uno a uno en la Capa Bronze para facilitar su procesamiento posterior.

In [0]:
import zipfile
import os

bronze_dir = "/Volumes/workspace/default/bronze_vuelos/Dataset"

print("Iniciando descompresión inteligente con nombres únicos...")

for file in os.listdir(bronze_dir):
    # Validamos que sea un archivo comprimido
    if file.endswith(".zip") or file.endswith(".zip.zip"):
        zip_path = os.path.join(bronze_dir, file)
        
        # Limpiamos el nombre para que el CSV final quede limpio (ej: flights_2025_01.csv)
        nombre_limpio = file.replace(".zip.zip", "").replace(".zip", "")
        nombre_csv_destino = f"{nombre_limpio}.csv"
        ruta_csv_destino = os.path.join(bronze_dir, nombre_csv_destino)
        
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            lista_archivos = zip_ref.namelist()
            # Buscamos el CSV interno sin importar cómo se llame de origen
            archivo_interno = [f for f in lista_archivos if f.endswith('.csv')][0]
            
            # Extraemos los bytes y los escribimos con el nombre del mes correspondiente
            with open(ruta_csv_destino, "wb") as f_out:
                f_out.write(zip_ref.read(archivo_interno))
                
        print(f" Extraído de forma independiente: {nombre_csv_destino}")



**Mostramos 5 lineas para ver el tipo de datos que tenemos**

In [0]:
# CELDA TEMPORAL DE INSPECCIÓN
# 1. Leemos una muestra pequeña de archivos crudos CSV
df_inspeccion = spark.read \
    .format("csv") \
    .option("header", "true") \
    .load("/Volumes/workspace/default/bronze_vuelos/Dataset/*.csv")

# 2. Imprimimos la lista exacta de nombres de columnas que detectó Spark
print("--- LISTA EXACTA DE COLUMNAS EN EL ARCHIVO ---")
print(df_inspeccion.columns)
print("-" * 45)

# 3. Mostramos las primeras 5 filas para ver cómo vienen formateados los datos
display(df_inspeccion.limit(5))


### Explicación de la Parte 1: Importación de librerías y definición del esquema permisivo

En esta sección se realiza lo siguiente:

1. **Importación de librerías:**  
   Se importan los módulos necesarios de PySpark para definir el esquema de los datos y aplicar funciones de transformación.

2. **Definición del esquema permisivo:**  
   Se crea un esquema (`StructType`) que describe la estructura y los tipos de datos esperados para cada columna del dataset de vuelos.  
   - Los nombres de las columnas se ajustan exactamente a los detectados en los archivos fuente (mayúsculas y guiones bajos) para evitar problemas de nulos por desalineación de nombres.
   - Todas las columnas se definen como tipo `StringType` y permisivas (`nullable=True`) para permitir la carga de datos incluso si hay valores inesperados o faltantes.
   - Este esquema facilita la ingesta inicial de datos crudos, permitiendo luego hacer conversiones de tipo y validaciones de calidad en pasos posteriores.

Esta parte es fundamental para asegurar que Spark lea correctamente todos los archivos CSV, sin perder información por diferencias en los nombres o tipos de columnas.

In [0]:
#-------------------------------------------------------------------------------------
# Parte 1: Importación de librerías y definición del esquema permisivo real de la BTS
#-------------------------------------------------------------------------------------
from pyspark.sql.types import StructType, StructField, StringType
import pyspark.sql.functions as F

# ESQUEMA DE LECTURA PERMISIVA REVISADO
# Explicación técnica: Ajustado con los nombres reales en mayúsculas y guiones bajos 
# detectados en la inspección visual del clúster para evitar nulos artificiales.
bts_permissive_schema = StructType([
    StructField("YEAR", StringType(), True),
    StructField("MONTH", StringType(), True),
    StructField("DAY_OF_MONTH", StringType(), True),
    StructField("DAY_OF_WEEK", StringType(), True),
    StructField("FL_DATE", StringType(), True),
    StructField("OP_UNIQUE_CARRIER", StringType(), True),
    StructField("OP_CARRIER_FL_NUM", StringType(), True),
    StructField("ORIGIN_AIRPORT_ID", StringType(), True),
    StructField("ORIGIN", StringType(), True),
    StructField("DEST_AIRPORT_ID", StringType(), True),
    StructField("DEST", StringType(), True),
    StructField("CRS_DEP_TIME", StringType(), True),
    StructField("DEP_TIME", StringType(), True),
    StructField("DEP_DELAY", StringType(), True),
    StructField("CRS_ARR_TIME", StringType(), True),
    StructField("ARR_DELAY", StringType(), True),
    StructField("CANCELLED", StringType(), True),
    StructField("CANCELLATION_CODE", StringType(), True),
    StructField("CRS_ELAPSED_TIME", StringType(), True),
    StructField("DISTANCE", StringType(), True),
    StructField("CARRIER_DELAY", StringType(), True),
    StructField("WEATHER_DELAY", StringType(), True),
    StructField("NAS_DELAY", StringType(), True),
    StructField("SECURITY_DELAY", StringType(), True),
    StructField("LATE_AIRCRAFT_DELAY", StringType(), True)
])



### Explicación de la Parte 2: Ingesta distribuida desde la Capa Bronze

En esta sección se realiza lo siguiente:

1. **Lectura distribuida de archivos CSV:**  
   Se utiliza Spark para leer todos los archivos CSV extraídos en la carpeta `bronze_vuelos/Dataset` de manera paralela y distribuida, aplicando el esquema permisivo definido previamente.

2. **Carga eficiente en memoria:**  
   Spark aprovecha la infraestructura distribuida para cargar todos los registros en memoria, permitiendo su procesamiento masivo y eficiente.

3. **Monitoreo de la ingesta:**  
   Se imprime el total de registros cargados para verificar que la ingesta fue exitosa y que no hubo pérdidas de datos en la lectura inicial.

Esta parte es fundamental para asegurar que todos los datos crudos estén disponibles en memoria y listos para su procesamiento y saneamiento en los siguientes pasos del pipeline.

In [0]:
#------------------------------------------------------------------
# Parte 2: Ingesta distribuida desde el volumen de la Capa Bronze 
#-----------------------------------------------------------------
bronze_path = "/Volumes/workspace/default/bronze_vuelos/Dataset"

# Leemos el directorio completo. Al estar bajo Databricks Serverless,
# Spark Connect abrirá y descomprimirá todos los .csv dentro de Dataset en paralelo.
df_raw_strings = spark.read \
    .format("csv") \
    .option("header", "true") \
    .schema(bts_permissive_schema) \
    .load(f"{bronze_path}/*.csv") # Agregamos /*.csv al final

# Monitoreo inicial de la ingesta real
print(f"Total de registros cargados en memoria distribuida (Capa Bronze): {df_raw_strings.count()}")

### Explicación de la Parte 3: Conversión, Duplicados y Calidad

En esta celda se realiza el procesamiento y saneamiento de los datos crudos de vuelos, siguiendo estos pasos principales:

1. **Conversión de Tipos:**  
   Se convierte cada columna del DataFrame de texto a su tipo de dato real (por ejemplo, enteros, dobles, fechas), usando `try_cast` y funciones de Spark para asegurar que los datos sean consistentes y analizables.

2. **Auditoría de Retrasos:**  
   Se audita la calidad de los datos de retraso, separando los vuelos puntuales (ArrDelay <= 15 min) y los retrasados críticos (ArrDelay > 15 min), verificando cuántos tienen información específica de causa de retraso.

3. **Eliminación de Duplicados y Lógica de Calidad:**  
   Se eliminan filas duplicadas y se crea una columna `is_corrupt` que marca los registros con datos esenciales nulos (año, mes, fecha de vuelo, aeropuertos, cancelación, distancia) para identificar registros corruptos o incompletos.

4. **Corrección de Nulos en Retrasos:**  
   Se rellenan con 0.0 los valores nulos en las columnas de desglose de retraso, asegurando que no haya valores vacíos en estos campos antes de guardar los datos.

Este procesamiento deja los datos listos para su partición y almacenamiento en la capa Silver, garantizando calidad y consistencia.

In [0]:
# ---------------------------------------------
# Parte 3 CONVERSIÓN MANUAL, DUPLICADOS Y CALIDAD
# ---------------------------------------------

# 1. Convertimos columna por columna de texto a su tipo real
df_casted = df_raw_strings.select(
    F.col("YEAR").try_cast("integer").alias("Year"),
    F.col("MONTH").try_cast("integer").alias("Month"),
    F.col("DAY_OF_MONTH").try_cast("integer").alias("DayofMonth"),
    F.col("DAY_OF_WEEK").try_cast("integer").alias("DayOfWeek"),
    F.to_date(F.col("FL_DATE"), "M/d/yyyy hh:mm:ss a").alias("FlightDate"), 
    F.col("OP_UNIQUE_CARRIER").cast("string").alias("Reporting_Airline"),
    F.col("OP_CARRIER_FL_NUM").cast("string").alias("Flight_Number_Reporting_Airline"),
    F.col("ORIGIN_AIRPORT_ID").try_cast("integer").alias("OriginAirportID"),
    F.col("ORIGIN").cast("string").alias("Origin"),
    F.col("DEST_AIRPORT_ID").try_cast("integer").alias("DestAirportID"),
    F.col("DEST").cast("string").alias("Dest"),
    F.col("CRS_DEP_TIME").try_cast("integer").alias("CRSDepTime"),
    F.col("DEP_TIME").try_cast("integer").alias("DepTime"),
    F.col("DEP_DELAY").try_cast("double").alias("DepDelay"),
    F.col("CRS_ARR_TIME").try_cast("integer").alias("CRSArrTime"),
    F.col("ARR_DELAY").try_cast("double").alias("ArrDelay"),
    F.col("CANCELLED").try_cast("double").alias("Cancelled"),
    F.col("CANCELLATION_CODE").cast("string").alias("CancellationCode"),
    F.col("CRS_ELAPSED_TIME").try_cast("double").alias("CRSElapsedTime"),
    F.col("DISTANCE").try_cast("double").alias("Distance"),
    F.col("CARRIER_DELAY").try_cast("double").alias("CarrierDelay"),
    F.col("WEATHER_DELAY").try_cast("double").alias("WeatherDelay"),
    F.col("NAS_DELAY").try_cast("double").alias("NASDelay"),
    F.col("SECURITY_DELAY").try_cast("double").alias("SecurityDelay"),
    F.col("LATE_AIRCRAFT_DELAY").try_cast("double").alias("LateAircraftDelay")
)

# --------------------------------------------------------------------------------------------------------
# AUDITORÍA: Verificación de la regla de nulos en desgloses de retraso
print(" Iniciando auditoría sobre las causas de retraso...")

df_puntuales = df_casted.filter(F.col("ArrDelay") <= 15)
total_puntuales = df_puntuales.count()
no_nulos_en_puntuales = df_puntuales.filter(F.col("CarrierDelay").isNotNull()).count()

print(f"\n[Muestra A: Vuelos con ArrDelay <= 15 min]")
print(f" -> Total de vuelos analizados: {total_puntuales:,}")
print(f" -> Vuelos con datos de retraso específico (No Nulos): {no_nulos_en_puntuales:,}")

df_retrasados_criticos = df_casted.filter(F.col("ArrDelay") > 15)
total_criticos = df_retrasados_criticos.count()
no_nulos_en_criticos = df_retrasados_criticos.filter(F.col("CarrierDelay").isNotNull()).count()

print(f"\n[Muestra B: Vuelos con ArrDelay > 15 min]")
print(f" -> Total de vuelos analizados: {total_criticos:,}")
print(f" -> Vuelos con datos de retraso específico (No Nulos): {no_nulos_en_criticos:,}")

#-------------------------------------------------------------------------------------------------------------------------------
# 2. Eliminamos duplicados a nivel de fila y aplicamos lógica de calidad basada en alias limpios
df_silver_base = df_casted.dropDuplicates().withColumn(
    "is_corrupt",
    F.when(
        F.col("Year").isNull() |
        F.col("Month").isNull() |
        F.col("FlightDate").isNull() |
        F.col("OriginAirportID").isNull() |
        F.col("DestAirportID").isNull() |
        F.col("Cancelled").isNull() |
        F.col("Distance").isNull(), 
        True
    ).otherwise(False)
)

# 3. CORRECCIÓN DE NULOS: Rellenamos con 0.0 las nuevas columnas antes de la partición
columnas_retraso = ["CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]
df_silver_base = df_silver_base.na.fill(value=0.0, subset=columnas_retraso)

### En la última celda se realiza la bifurcación y escritura de los datos procesados en tres tablas Delta diferentes: una para registros rechazados por calidad, otra para cancelaciones, y otra para retrasos, cada una con su lógica de particionado. Esto permite organizar y almacenar los datos limpios y auditados en la Capa Silver para su análisis posterior.

In [0]:
# ---------------------------------------------
# Part 4: BIFURCACIÓN Y ESCRITURA DELTA
# -------------------------------------------

# TABLA 1: Rechazados (is_corrupt es True) -> Sin particionar
df_rechazados = df_silver_base.filter(F.col("is_corrupt") == True)
df_rechazados.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_vuelos_rechazados")

# TABLA 2: Cancelaciones (is_corrupt es False) -> Particionado por Año/Mes
df_cancelaciones = df_silver_base.filter(F.col("is_corrupt") == False)
df_cancelaciones.write.format("delta").mode("overwrite").partitionBy("Year", "Month").saveAsTable("workspace.default.silver_vuelos_cancelaciones")

# TABLA 3: Retrasos (Datos limpios Y que NO se hayan cancelado) -> Particionado por Año/Mes
df_retrasos = df_cancelaciones.filter(F.col("Cancelled") == 0)
df_retrasos.write.format("delta").mode("overwrite").partitionBy("Year", "Month").saveAsTable("workspace.default.silver_vuelos_retrasos")

print("¡Pipeline Bronze a Silver completado con éxito y tablas operacionales llenas con desgloses sanitizados!")